In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Load Titanic dataset
df = sns.load_dataset("titanic")

# Display first 5 rows
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [3]:
import pandas as pd

df = pd.read_csv("../data_pipeline/books_cleaned.csv")

print("Dataset loaded successfully!")
print("Dataset Shape:", df.shape)

Dataset loaded successfully!
Dataset Shape: (100, 6)


In [4]:
# Dataset shape
print("Dataset Shape:", df.shape)

# Column names
print("\nColumn Names:")
print(df.columns.tolist())

# Data types
print("\nData Types:")
print(df.dtypes)

Dataset Shape: (100, 6)

Column Names:
['title', 'price_gbp', 'price_inr', 'rating', 'in_stock', 'category']

Data Types:
title         object
price_gbp    float64
price_inr    float64
rating         int64
in_stock        bool
category      object
dtype: object


In [5]:
# Missing values count
missing_count = df.isnull().sum()

# Missing values percentage
missing_percentage = (df.isnull().mean() * 100).round(2)

missing_summary = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing Percentage": missing_percentage
})

missing_summary

,Missing Count,Missing Percentage
title,0,0.0
price_gbp,0,0.0
price_inr,0,0.0
rating,0,0.0
in_stock,0,0.0
category,0,0.0


In [6]:
# Apply 30% missingness threshold

missing_threshold = 30

# Check columns with more than 30% missing values
high_missing_cols = missing_summary[
    missing_summary["Missing Percentage"] > missing_threshold
].index.tolist()

print("Columns dropped due to high missingness:")
print(high_missing_cols)

# Create cleaned dataset
df_clean = df.drop(columns=high_missing_cols)

print("\nRemaining columns:")
print(df_clean.columns.tolist())

Columns dropped due to high missingness:
[]

Remaining columns:
['title', 'price_gbp', 'price_inr', 'rating', 'in_stock', 'category']


In [7]:
import seaborn as sns

# Load Titanic dataset
df = sns.load_dataset("titanic")

print("Titanic dataset loaded successfully!")
print("Dataset Shape:", df.shape)

# Define features and target
features = ['pclass', 'sex', 'age', 'fare', 'embarked']
target = 'survived'

X = df[features]
y = df[target]

print("\nFeatures:")
print(X.columns.tolist())

print("\nTarget:")
print(target)

print("\nFeature Shape:", X.shape)
print("Target Shape:", y.shape)

Titanic dataset loaded successfully!
Dataset Shape: (891, 15)

Features:
['pclass', 'sex', 'age', 'fare', 'embarked']

Target:
survived

Feature Shape: (891, 5)
Target Shape: (891,)


In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (712, 5)
X_test shape: (179, 5)
y_train shape: (712,)
y_test shape: (179,)


In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

# Separate numerical and categorical features
numeric_features = ['pclass', 'age', 'fare']
categorical_features = ['sex', 'embarked']

# Numerical preprocessing
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical preprocessing
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combine preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

# Logistic Regression pipeline
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

# Train model
model.fit(X_train, y_train)

print("Logistic Regression model trained successfully!")

Logistic Regression model trained successfully!


In [14]:
# Make predictions on the test set
y_pred = model.predict(X_test)

print("Predictions generated successfully.")
print("First 10 predictions:", y_pred[:10])

Predictions generated successfully.
First 10 predictions: [0 0 0 0 1 0 1 0 0 0]


In [19]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

accuracy = accuracy_score(y_test, y_pred)

print("Model Accuracy:", accuracy)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Model Accuracy: 0.770949720670391

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.84      0.82       110
           1       0.72      0.67      0.69        69

    accuracy                           0.77       179
   macro avg       0.76      0.75      0.75       179
weighted avg       0.77      0.77      0.77       179


Confusion Matrix:
[[92 18]
 [23 46]]


### Model Evaluation Interpretation

The model achieved an accuracy of approximately 77.09% on the test dataset.

The classification report shows balanced performance across both classes, with an F1-score of 0.82 for Class 0 and 0.69 for Class 1.

The confusion matrix shows that the model correctly classified 92 samples from Class 0 and 46 samples from Class 1, while 18 and 23 samples were misclassified respectively.

Overall, the model provides a reasonable baseline performance for the classification task.

In [15]:
# Make predictions using the trained model
predictions = model.predict(X_test)

print("Predictions generated successfully!")
print("First 10 predictions:", predictions[:10])

Predictions generated successfully!
First 10 predictions: [0 0 0 0 1 0 1 0 0 0]


## Prediction

### Prediction Interpretation

The trained Logistic Regression model successfully generated predictions for the test dataset.

The first 10 predictions show the predicted class labels for the test samples.

These predictions can be compared with the actual target values to evaluate the model's predictive performance.

# Module 2 Summary

In this module, exploratory data analysis and machine learning modeling were performed on the dataset.

The data was prepared by handling missing values and separating numerical and categorical features. A preprocessing pipeline was created using imputation, scaling, and one-hot encoding.

A Logistic Regression model was trained using the training dataset and evaluated on the test dataset. The model achieved an accuracy of approximately 77.09%.

The classification report and confusion matrix were used to evaluate the model performance. Predictions were also generated for the test dataset.

Overall, the modeling workflow demonstrates the complete process from data preprocessing to model training, evaluation, and prediction.